### A7 Step 1: Stability (PSI, 2013–14 vs 2015)

PSI measures how much a feature's distribution shifted: bins learned on 2013–14 (deciles for numeric fields, levels for categories), then PSI = Σ (new% − old%) × ln(new% / old%). Rule of thumb: < 0.10 stable, 0.10–0.25 moderate, > 0.25 major.
- All features and factor scores are stable (PSI < 0.10) except job_family (0.128).
- PSI tracks the bulk of the distribution. The A3 finding (more 2015 loans above the 2014 caps for revol_bal / total_rev_hi_lim) sits in the top 0.5% and barely moves any decile, so monitoring needs both views.
- **job_family**: its unmatched ("other") share dropped from 66–67% (2013Q1–Q3) to 16–17% from 2013Q4 and stayed flat, while blank titles stayed at 4–7%. A step like this is a change in what the field contains (apparently employer names → job titles), not in who borrowed. The training window mixes both eras, so job_family is excluded (definition unstable over time; also a fairness proxy risk and weak). 29 features remain.
- Holdout (2016) PSI follows once the pipeline is built (A8).

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

train = pd.read_parquet(Path("../data/interim/train_a6.parquet"))
SPEC = json.load(open("../reports/a6_spec.json"))
FIT_CUTOFF = pd.Timestamp("2014-12-31")
TARGET = "defaulted"
is_fit = train["issue_dt"] <= FIT_CUTOFF
fit, later = train[is_fit], train[~is_fit]
print(fit.shape, later.shape, len(SPEC["features"]))

(14373, 85) (15627, 85) 30


In [2]:
def psi(expected, actual, n_bins=10, eps=1e-4):
    """Population Stability Index: how much `actual` shifted from `expected` (bins learned on expected)."""
    if pd.api.types.is_numeric_dtype(expected) and expected.nunique() > 12:
        edges = np.unique(np.quantile(expected.dropna(), np.linspace(0, 1, n_bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        to_bin = lambda s: pd.Series(pd.cut(s, edges, labels=False), index=s.index).fillna(-1).astype(int)   # -1 = missing
    else:
        to_bin = lambda s: s.astype(str).where(s.notna(), "missing")
    old = to_bin(expected).value_counts(normalize=True)
    new = to_bin(actual).value_counts(normalize=True)
    bins = old.index.union(new.index)
    old, new = old.reindex(bins, fill_value=0).clip(lower=eps), new.reindex(bins, fill_value=0).clip(lower=eps)
    return float(((new - old) * np.log(new / old)).sum())

def psi_label(v):
    return "stable" if v < 0.10 else ("moderate shift" if v < 0.25 else "MAJOR SHIFT")

cols = SPEC["features"] + SPEC["fa"]["score_columns"]
psi_table = pd.DataFrame({"psi": [round(psi(fit[c], later[c]), 3) for c in cols]}, index=cols)
psi_table["verdict"] = psi_table["psi"].apply(psi_label)
display(psi_table.sort_values("psi", ascending=False))

,psi,verdict
job_family,0.128,moderate shift
inq_last_6mths,0.045,stable
num_tl_op_past_12m,0.035,stable
dti_cap,0.025,stable
revol_util_cap,0.021,stable
acc_open_past_24mths,0.021,stable
bc_util_cap,0.020,stable
factor_credit_seeking,0.019,stable
factor_card_utilization,0.017,stable
open_acc,0.012,stable


In [3]:
print("job_family PSI:", round(psi(fit["job_family"], later["job_family"]), 3))

shares = pd.DataFrame({
    "share_2013_14": fit["job_family"].value_counts(normalize=True),
    "share_2015": later["job_family"].value_counts(normalize=True),
}).round(3)
shares["change"] = (shares["share_2015"] - shares["share_2013_14"]).round(3)
display(shares.sort_values("change", key=abs, ascending=False))

print("emp_title missing 2013-14 / 2015:",
      round(fit["emp_title"].isna().mean(), 3), "/", round(later["emp_title"].isna().mean(), 3))

job_family PSI: 0.128


,share_2013_14,share_2015,change
job_family,,,
other,0.287,0.163,-0.124
management,0.142,0.190,0.048
finance_admin,0.123,0.154,0.031
transport_trades,0.083,0.106,0.023
public_safety_gov,0.035,0.016,-0.019
sales_service,0.059,0.076,0.017
tech_engineering,0.073,0.086,0.013
owner_self,0.011,0.022,0.011
healthcare,0.071,0.064,-0.007


emp_title missing 2013-14 / 2015: 0.057 / 0.062


In [4]:
q = train.groupby(train["issue_dt"].dt.to_period("Q"))
by_quarter = pd.DataFrame({
    "loans": q.size(),
    "other_share": q["job_family"].apply(lambda s: (s == "other").mean()).round(3),
    "unknown_share": q["job_family"].apply(lambda s: (s == "unknown").mean()).round(3),
})
display(by_quarter)

,loans,other_share,unknown_share
issue_dt,,,
2013Q1,893,0.667,0.060
2013Q2,1246,0.669,0.052
2013Q3,1474,0.660,0.065
2013Q4,1736,0.172,0.063
2014Q1,1906,0.161,0.070
2014Q2,2091,0.157,0.056
2014Q3,2336,0.157,0.038
2014Q4,2691,0.159,0.060
2015Q1,3249,0.168,0.061


In [5]:
SPEC["features"] = [c for c in SPEC["features"] if c != "job_family"]
SPEC["exclude"]["job_family"] = (
    "definition changed mid-sample: unmatched ('other') share 66-67% in 2013Q1-Q3, then 16-17% from 2013Q4 "
    "(emp_title appears to switch from employer names to job titles); blank share stable. "
    "PSI vs 2015 = 0.128 (understated, since the fit window mixes both eras). Also a fairness proxy risk and weak signal.")
SPEC.get("fairness_review", {}).pop("job_family", None)
SPEC.setdefault("stability", {})["psi_2013_14_vs_2015"] = psi_table["psi"].to_dict()
SPEC["stability"]["note"] = ("All features stable (PSI < 0.10) except job_family (0.128). PSI tracks the bulk; "
                             "tail growth in revol_bal / total_rev_hi_lim (A3 cap counts) is not visible in PSI.")
with open("../reports/a7_spec.json", "w") as f:
    json.dump(SPEC, f, indent=2)
print("features:", len(SPEC["features"]))          # expect 29

features: 29


### A7 Step 2: Confidence intervals for default rates (loans ≤ 2014)

A default rate from our loans is a prediction for future borrowers, so each is reported with a 95% Wilson interval (the range likely to contain the true rate; it stays within 0–100% and suits small groups). Non-overlapping ranges = clearly different groups.
- **Term**: 36m 13.2% (12.6–13.9) vs 60m 28.3% (26.9–29.7).
- **Grade**: A–E clearly separated steps; E/F/G overlap (small groups; G's range is 20 pts wide).
- **Break-even (12.5%)**: grades A (≤ 5.4%) and B (≤ 11.8%) are profitable on average; C and worse are not; even 36-month loans overall sit just above break-even (12.6–13.9%). The model's value lies in finding good borrowers inside the risky groups.
- **Home ownership**: mortgage 15.4% vs rent 19.8% clearly different; own (15.2–19.1) overlaps both.
- **Verification**: 12.7% vs 19.1%, clearly different (confounded, A4).
- **Purpose**: small_business (23.2–38.3) clearly riskier than debt_consolidation (17.4–19.1); credit_card clearly safer; car, medical and major_purchase have 10–13 pt wide ranges, supporting the A5 risk-level merge.
Saved: reports/a7_default_rate_ci.csv.

In [6]:
from statsmodels.stats.proportion import proportion_confint

def rate_ci(df, col, target=TARGET):
    """Default rate per group with a 95% Wilson confidence interval."""
    g = df.groupby(col)[target]
    defaults, loans = g.sum(), g.size()
    low, high = proportion_confint(defaults, loans, alpha=0.05, method="wilson")
    return pd.DataFrame({"loans": loans, "defaults": defaults,
                         "rate": (defaults / loans).round(3),
                         "ci_low": low.round(3), "ci_high": high.round(3),
                         "width": (high - low).round(3)})

for col in ["term_months", "grade", "home_ownership", "verification_grp", "purpose_grp"]:
    print(f"=== {col} ===")
    display(rate_ci(fit, col))

=== term_months ===


,loans,defaults,rate,ci_low,ci_high,width
term_months,,,,,,
36,10545,1396,0.132,0.126,0.139,0.013
60,3828,1082,0.283,0.269,0.297,0.029


=== grade ===


,loans,defaults,rate,ci_low,ci_high,width
grade,,,,,,
A,2162,97,0.045,0.037,0.054,0.018
B,4131,447,0.108,0.099,0.118,0.019
C,4069,773,0.190,0.178,0.202,0.024
D,2390,589,0.246,0.230,0.264,0.035
E,1122,372,0.332,0.305,0.360,0.055
F,406,155,0.382,0.336,0.430,0.094
G,93,45,0.484,0.385,0.584,0.199


=== home_ownership ===


,loans,defaults,rate,ci_low,ci_high,width
home_ownership,,,,,,
MORTGAGE,7407,1137,0.154,0.145,0.162,0.016
OWN,1406,240,0.171,0.152,0.191,0.039
RENT,5560,1101,0.198,0.188,0.209,0.021


=== verification_grp ===


,loans,defaults,rate,ci_low,ci_high,width
verification_grp,,,,,,
not_verified,4257,541,0.127,0.117,0.137,0.020
verified,10116,1937,0.191,0.184,0.199,0.015


=== purpose_grp ===


,loans,defaults,rate,ci_low,ci_high,width
purpose_grp,,,,,,
car,106,15,0.142,0.088,0.220,0.133
credit_card,3463,505,0.146,0.134,0.158,0.024
debt_consolidation,8616,1571,0.182,0.174,0.191,0.016
home_improvement,794,122,0.154,0.130,0.180,0.050
major_purchase,223,34,0.152,0.111,0.205,0.094
medical,140,28,0.200,0.142,0.274,0.132
other,892,161,0.180,0.157,0.207,0.050
small_business,139,42,0.302,0.232,0.383,0.151


In [7]:
ci_tables = pd.concat({col: rate_ci(fit, col) for col in
                       ["term_months", "grade", "home_ownership", "verification_grp", "purpose_grp"]},
                      names=["grouping", "level"])
ci_tables.to_csv("../reports/a7_default_rate_ci.csv")
print(ci_tables.shape)

(22, 6)


### A7 Step 3: Factors vs grade

Grade is Lending Club's risk model (benchmark, not an input). Our 4 factors use only credit-bureau data.
- **Average factor score by grade** (0 = average borrower): card utilization rises from −0.76 (A) to +0.52 (G), Spearman 0.361 (strongest link); credit-seeking −0.27 → +0.46 (0.188); debt size high only for grade A (+0.47; Spearman −0.186); card size ≈ no relation (−0.086). Grade A is a distinct profile: low utilization, few new accounts, established credit files.
- **Predictive power** (AUC; Gini = 2·AUC − 1): grade 0.677 (2013–14) / 0.703 (2015); 4 factors combined in a logistic regression trained on 2013–14: 0.632 / 0.633. The factors capture about 75% of grade's Gini (0.264 vs 0.354) and are stable on unseen 2015 loans. Grade's 2015 rise may reflect model improvements or the in-progress label bias (A2); not separable here.
- **Weights**: credit-seeking +0.377, card utilization +0.300, debt size −0.238, card size +0.043. Grade aligns most with utilization, but credit-seeking carries the most default signal: possibly under-used by grade, to check in Part B.
- The remaining gap (term, FICO, loan_to_income, DTI, inquiries, data we don't have) is what the Part B model must close.

In [8]:
GRADE_NUM = {g: i for i, g in enumerate("ABCDEFG", start=1)}     # A=1 ... G=7
score_cols = SPEC["fa"]["score_columns"]

fit_g = fit.assign(grade_num=fit["grade"].map(GRADE_NUM))
display(fit_g.groupby("grade")[score_cols].mean().round(2))

spearman = fit_g[score_cols].corrwith(fit_g["grade_num"], method="spearman").round(3)
print("Spearman correlation with grade (A=1 ... G=7):")
print(spearman)

,factor_debt_size,factor_card_utilization,factor_card_size,factor_credit_seeking
grade,,,,
A,0.47,-0.76,0.26,-0.27
B,0.04,-0.13,0.00,-0.10
C,-0.12,0.18,-0.07,0.02
D,-0.21,0.32,-0.10,0.18
E,-0.11,0.40,0.01,0.30
F,-0.18,0.51,-0.14,0.29
G,-0.02,0.52,0.07,0.46


Spearman correlation with grade (A=1 ... G=7):
factor_debt_size          -0.186
factor_card_utilization    0.361
factor_card_size          -0.086
factor_credit_seeking      0.188
dtype: float64


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

later_g = later.assign(grade_num=later["grade"].map(GRADE_NUM))

lr = LogisticRegression().fit(fit[score_cols], fit[TARGET])      # learn weights on 2013-14 only

comparison = pd.DataFrame({
    "auc_2013_14": [roc_auc_score(fit_g[TARGET], fit_g["grade_num"]),
                    roc_auc_score(fit[TARGET], lr.predict_proba(fit[score_cols])[:, 1])],
    "auc_2015": [roc_auc_score(later_g[TARGET], later_g["grade_num"]),
                 roc_auc_score(later[TARGET], lr.predict_proba(later[score_cols])[:, 1])],
}, index=["grade (old model)", "4 factor scores combined"]).round(3)
display(comparison)
print("factor weights:", dict(zip(score_cols, lr.coef_[0].round(3))))

,auc_2013_14,auc_2015
grade (old model),0.677,0.703
4 factor scores combined,0.632,0.633


factor weights: {'factor_debt_size': np.float64(-0.238), 'factor_card_utilization': np.float64(0.3), 'factor_card_size': np.float64(0.043), 'factor_credit_seeking': np.float64(0.377)}


In [10]:
weights = pd.Series(lr.coef_[0], index=score_cols).round(3)
print(weights)

SPEC.setdefault("factors_vs_grade", {}).update({
    "spearman_with_grade": spearman.to_dict(),
    "auc": comparison.to_dict(),
    "gini_2013_14": {"grade": round(2 * comparison.loc["grade (old model)", "auc_2013_14"] - 1, 3),
                     "factors": round(2 * comparison.loc["4 factor scores combined", "auc_2013_14"] - 1, 3)},
    "logit_weights": weights.to_dict(),
})
with open("../reports/a7_spec.json", "w") as f:
    json.dump(SPEC, f, indent=2)
print("saved")

factor_debt_size          -0.238
factor_card_utilization    0.300
factor_card_size           0.043
factor_credit_seeking      0.377
dtype: float64
saved


### A7 Step 4: Location as a possible proxy

A proxy is a feature that is not itself protected but is linked to a protected characteristic (the historical example is redlining: credit refused by neighbourhood). The data has no protected attributes, so proxy effects cannot be measured; state therefore had to show a clear, stable benefit (rule set in advance: out-of-time AUC ≥ 0.53).
- 16 states with ≥ 300 loans range 13.4% (GA) to 20.0% (NY); nearly every 95% range includes the overall 17.2%. With 16 comparisons, ~1 borderline state is expected by chance.
- States with < 100 loans merged into "other" (32 kept). WoE learned on 2013–14 → 2015 AUC **0.5116**.
- Chi-square across the 16 big states: p = 0.030. Differences are detectable but not useful for ranking new loans (statistically significant ≠ useful).
- **addr_state_clean excluded.** The model has no location features (zip_code excluded in A4). Limitation: indirect discrimination cannot be tested with this data; a deployment would need a disparate-impact audit.

In [11]:
from statsmodels.stats.proportion import proportion_confint

state = rate_ci(fit, "addr_state_clean").sort_values("loans", ascending=False)
print("states:", len(state), "| states with < 100 loans:", int((state["loans"] < 100).sum()))
print("\nLargest 5 states:")
display(state.head(5))
print("\nStates with >= 300 loans, sorted by default rate:")
display(state[state["loans"] >= 300].sort_values("rate"))

states: 47 | states with < 100 loans: 15

Largest 5 states:


,loans,defaults,rate,ci_low,ci_high,width
addr_state_clean,,,,,,
CA,2129,379,0.178,0.162,0.195,0.032
NY,1150,230,0.200,0.178,0.224,0.046
TX,1144,175,0.153,0.133,0.175,0.042
FL,920,164,0.178,0.155,0.204,0.049
IL,601,92,0.153,0.127,0.184,0.058



States with >= 300 loans, sorted by default rate:


,loans,defaults,rate,ci_low,ci_high,width
addr_state_clean,,,,,,
GA,462,62,0.134,0.106,0.168,0.062
CO,310,43,0.139,0.105,0.182,0.077
WA,339,50,0.147,0.114,0.189,0.075
TX,1144,175,0.153,0.133,0.175,0.042
IL,601,92,0.153,0.127,0.184,0.058
VA,418,64,0.153,0.122,0.191,0.069
OH,448,72,0.161,0.130,0.198,0.068
MD,348,60,0.172,0.136,0.216,0.079
CA,2129,379,0.178,0.162,0.195,0.032


In [12]:
from scipy.stats import chi2_contingency

def woe_map(df, col, target=TARGET):
    """{group: WoE} learned from df (missing as its own group, +0.5 smoothing)."""
    groups = df[col].astype(str).where(df[col].notna(), "missing")
    t = pd.crosstab(groups, df[target])
    good, bad = t[0] + 0.5, t[1] + 0.5
    return np.log((good / good.sum()) / (bad / bad.sum())).to_dict()

def out_of_time_auc(fit_df, later_df, col, target=TARGET):
    """AUC on later data using WoE values learned on fit data only; unseen group -> 0."""
    groups = later_df[col].astype(str).where(later_df[col].notna(), "missing")
    woe = groups.map(woe_map(fit_df, col)).fillna(0.0)
    return round(roc_auc_score(later_df[target], -woe), 4)

# 1. merge states with < 100 loans in 2013-14 into "other"
counts = fit["addr_state_clean"].value_counts()
STATE_KEEP = sorted(counts[counts >= 100].index.tolist())
for df in (train, fit, later):
    df["state_grp"] = df["addr_state_clean"].where(df["addr_state_clean"].isin(STATE_KEEP), "other")
print("states kept:", len(STATE_KEEP), "| loans in 'other' (2013-14):", int((fit["state_grp"] == "other").sum()))

# 2. fair test: WoE learned on 2013-14, AUC on 2015
print("2015 AUC with 2013-14 state WoE:", out_of_time_auc(fit, later, "state_grp"))

# 3. chi-square across states with >= 300 loans: could the differences be luck?
big = fit[fit["addr_state_clean"].isin(counts[counts >= 300].index)]
print("chi-square p (16 big states):", round(chi2_contingency(pd.crosstab(big["addr_state_clean"], big[TARGET]))[1], 4))

states kept: 32 | loans in 'other' (2013-14): 741
2015 AUC with 2013-14 state WoE: 0.5116
chi-square p (16 big states): 0.0304


In [13]:
SPEC["pending"].pop("addr_state_clean", None)
SPEC["exclude"]["addr_state_clean"] = (
    "location proxy risk (cannot be tested: no protected attributes in the data) with negligible predictive value: "
    "2015 AUC with 2013-14 WoE 0.5116 (rule: < 0.53 -> exclude); chi-square across 16 big states p = 0.030 "
    "(differences detectable in 2013-14 but not useful for ranking new loans)")
SPEC["notes"].append("No location features in the model (zip_code excluded in A4, state in A7). "
                     "Indirect discrimination cannot be tested without protected attributes; "
                     "a production deployment would need a disparate-impact audit.")
with open("../reports/a7_spec.json", "w") as f:
    json.dump(SPEC, f, indent=2)
print("pending:", SPEC["pending"], "| features:", len(SPEC["features"]))     # expect {} | 29

pending: {} | features: 29


In [14]:
print("fico_mid  min / 1st percentile:", fit["fico_mid"].min(), "/", fit["fico_mid"].quantile(0.01))
print("dti       max (raw):            ", round(fit["dti"].max(), 2))
print("share with any public record:   ", round((fit["pub_rec"] > 0).mean(), 3))
print("share with a bankruptcy:        ", round((fit["pub_rec_bankruptcies"] > 0).mean(), 3))
print("share with delinquency in 2 yrs:", round((fit["delinq_2yrs"] > 0).mean(), 3))

fico_mid  min / 1st percentile: 662.0 / 662.0
dti       max (raw):             39.97
share with any public record:    0.158
share with a bankruptcy:         0.124
share with delinquency in 2 yrs: 0.195
